In [1]:
# ===== READ SILVER TABLES =====

import com.microsoft.spark.fabric

sales = spark.read.synapsesql("Retail360_Silver.dbo.sales")
customers = spark.read.synapsesql("Retail360_Silver.dbo.customers")
products = spark.read.synapsesql("Retail360_Silver.dbo.products")
stores = spark.read.synapsesql("Retail360_Silver.dbo.stores")

print("Silver tables loaded successfully!")

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 3, Finished, Available, Finished, False)

Silver tables loaded successfully!


In [2]:
# ===== GOLD: SALES WITH CUSTOMER DETAILS =====

gold_sales = (
    sales
    .join(
        customers,
        sales.customer_id == customers.customer_id,
        "left"
    )
    .select(
        sales["*"],
        customers["customer_name"],
        customers["city"].alias("customer_city"),
        customers["segment"]
    )
)

gold_sales.show(5)

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 4, Finished, Available, Finished, False)

+--------+--------+-----------+--------+------------+------------+--------------+----------+----------+---------------+----------+----------+-------------+-------------+--------------+
|quantity|order_id|customer_id|store_id|gross_amount|discount_pct|payment_method|unit_price|order_date|discount_amount|product_id|net_amount|customer_name|customer_city|       segment|
+--------+--------+-----------+--------+------------+------------+--------------+----------+----------+---------------+----------+----------+-------------+-------------+--------------+
|       3|O1000005|    C100296|    S105|    53199.57|           0|   NET BANKING|  17733.19|2025-10-13|            0.0|     P1004|  53199.57| Customer_296|    AHMEDABAD|      CONSUMER|
|       3|O1000003|    C100316|    S113|    22085.07|          10|   NET BANKING|   7361.69|2025-04-03|        2208.51|     P1084|  19876.56| Customer_316|      KOLKATA|      CONSUMER|
|       5|O1000001|    C100427|    S108|    43559.15|           5|         

In [3]:
# ===== GOLD: ADD PRODUCT & STORE DETAILS =====

gold_sales = (
    gold_sales
    .join(
        products,
        gold_sales.product_id == products.product_id,
        "left"
    )
    .join(
        stores,
        gold_sales.store_id == stores.store_id,
        "left"
    )
    .select(
        gold_sales["*"],
        products["product_name"],
        products["category"].alias("product_category"),
        stores["store_name"],
        stores["city"].alias("store_city"),
        stores["store_type"]
    )
)

gold_sales.show(5)

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 5, Finished, Available, Finished, False)

+--------+--------+-----------+--------+------------+------------+--------------+----------+----------+---------------+----------+----------+-------------+-------------+--------------+------------+----------------+------------------+----------+-----------+
|quantity|order_id|customer_id|store_id|gross_amount|discount_pct|payment_method|unit_price|order_date|discount_amount|product_id|net_amount|customer_name|customer_city|       segment|product_name|product_category|        store_name|store_city| store_type|
+--------+--------+-----------+--------+------------+------------+--------------+----------+----------+---------------+----------+----------+-------------+-------------+--------------+------------+----------------+------------------+----------+-----------+
|       1|O1000004|    C100486|    S114|      1903.0|           5|   NET BANKING|    1903.0|2025-08-14|          95.15|     P1080|   1807.85| Customer_486|    BENGALURU|SMALL BUSINESS|  Product_80|         FASHION|Retail360 Store

In [4]:
# ===== SAVE FINAL GOLD SALES TABLE =====

gold_sales.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_sales")

print("Gold sales table saved successfully!")

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 6, Finished, Available, Finished, False)

Gold sales table saved successfully!


In [5]:
from pyspark.sql import functions as F

daily_sales = (
    gold_sales
    .groupBy("order_date")
    .agg(
        F.sum("net_amount").alias("total_sales"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy("order_date")
)

daily_sales.show(10)

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 7, Finished, Available, Finished, False)

+----------+------------------+--------------+------------+----------------+
|order_date|       total_sales|total_quantity|total_orders|unique_customers|
+----------+------------------+--------------+------------+----------------+
|2025-01-01|         820597.23|            62|          25|              24|
|2025-01-02|1166289.6300000001|            83|          25|              23|
|2025-01-03| 746560.4499999998|            70|          26|              23|
|2025-01-04|         847252.25|            84|          29|              27|
|2025-01-05| 878044.7299999999|            87|          28|              28|
|2025-01-06|        1095812.12|            92|          31|              31|
|2025-01-07|        1054881.78|           103|          31|              29|
|2025-01-08| 920484.8599999999|            80|          28|              27|
|2025-01-09|        1177886.04|            89|          31|              30|
|2025-01-10|1110235.1400000001|           107|          35|              32|

In [6]:
category_sales = (
    gold_sales
    .groupBy("product_category")
    .agg(
        F.sum("net_amount").alias("total_sales"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy(F.desc("total_sales"))
)

category_sales.show()

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 8, Finished, Available, Finished, False)

+----------------+--------------------+--------------+------------+
|product_category|         total_sales|total_quantity|total_orders|
+----------------+--------------------+--------------+------------+
|         GROCERY|  7.69846303399998E7|          6652|        2162|
|     ELECTRONICS| 7.451749848999985E7|          5399|        1766|
|          BEAUTY| 7.188629026000006E7|          7697|        2543|
|  HOME & KITCHEN| 6.500411638000016E7|          5869|        1927|
|         FASHION|5.6828034910000116E7|          4781|        1599|
+----------------+--------------------+--------------+------------+



In [7]:
store_sales = (
    gold_sales
    .groupBy("store_id", "store_name", "store_city", "store_type")
    .agg(
        F.sum("net_amount").alias("total_sales"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy(F.desc("total_sales"))
)

store_sales.show(10)


StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 9, Finished, Available, Finished, False)

+--------+------------------+----------+-----------+--------------------+--------------+------------+----------------+
|store_id|        store_name|store_city| store_type|         total_sales|total_quantity|total_orders|unique_customers|
+--------+------------------+----------+-----------+--------------------+--------------+------------+----------------+
|    S116|Retail360 Store 16|     DELHI|HIGH STREET| 1.881785772000001E7|          1693|         549|             334|
|    S114|Retail360 Store 14|     DELHI|HIGH STREET| 1.845600435000001E7|          1573|         505|             331|
|    S113|Retail360 Store 13| HYDERABAD|       MALL| 1.831482487000001E7|          1610|         516|             314|
|    S115|Retail360 Store 15| BENGALURU|HIGH STREET|1.8086462950000018E7|          1582|         519|             326|
|    S102| Retail360 Store 2|    MUMBAI|       MALL|1.7947306550000004E7|          1558|         515|             313|
|    S105| Retail360 Store 5|    MUMBAI|     OUT

In [8]:
segment_sales = (
    gold_sales
    .groupBy("segment")
    .agg(
        F.sum("net_amount").alias("total_sales"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.avg("net_amount").alias("average_order_value")
    )
    .orderBy(F.desc("total_sales"))
)

segment_sales.show()

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 10, Finished, Available, Finished, False)

+--------------+-------------------+--------------+------------+----------------+-------------------+
|       segment|        total_sales|total_quantity|total_orders|unique_customers|average_order_value|
+--------------+-------------------+--------------+------------+----------------+-------------------+
|      CONSUMER|2.299104222299998E8|         20360|        6676|             334| 34438.349644996975|
|     CORPORATE|6.112223119000007E7|          5294|        1762|              88|   34689.1209931896|
|SMALL BUSINESS|5.418791696000007E7|          4744|        1559|              78|  34758.12505452217|
+--------------+-------------------+--------------+------------+----------------+-------------------+



In [9]:
# ===== SAVE GOLD KPI TABLES =====

daily_sales.write.mode("overwrite").format("delta").saveAsTable("daily_sales")
category_sales.write.mode("overwrite").format("delta").saveAsTable("category_sales")
store_sales.write.mode("overwrite").format("delta").saveAsTable("store_sales")
segment_sales.write.mode("overwrite").format("delta").saveAsTable("segment_sales")

print("All Gold KPI tables saved successfully!")

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 11, Finished, Available, Finished, False)

All Gold KPI tables saved successfully!


In [10]:
# ===== VERIFY GOLD KPI TABLES =====

print("===== GOLD TABLES =====")

spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 12, Finished, Available, Finished, False)

===== GOLD TABLES =====
+------------------------------+--------------+-----------+
|namespace                     |tableName     |isTemporary|
+------------------------------+--------------+-----------+
|Retail360.Retail360_Silver.dbo|category_sales|false      |
|Retail360.Retail360_Silver.dbo|customers     |false      |
|Retail360.Retail360_Silver.dbo|daily_sales   |false      |
|Retail360.Retail360_Silver.dbo|gold_sales    |false      |
|Retail360.Retail360_Silver.dbo|overall_kpi   |false      |
|Retail360.Retail360_Silver.dbo|products      |false      |
|Retail360.Retail360_Silver.dbo|sales         |false      |
|Retail360.Retail360_Silver.dbo|segment_sales |false      |
|Retail360.Retail360_Silver.dbo|store_sales   |false      |
|Retail360.Retail360_Silver.dbo|stores        |false      |
+------------------------------+--------------+-----------+



In [11]:
# ===== GOLD KPI VALIDATION =====

print("Daily Sales:")
spark.read.table("daily_sales").show(5, truncate=False)

print("Category Sales:")
spark.read.table("category_sales").show(5, truncate=False)

print("Store Sales:")
spark.read.table("store_sales").show(5, truncate=False)

print("Segment Sales:")
spark.read.table("segment_sales").show(5, truncate=False)

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 13, Finished, Available, Finished, False)

Daily Sales:
+----------+------------------+--------------+------------+----------------+
|order_date|total_sales       |total_quantity|total_orders|unique_customers|
+----------+------------------+--------------+------------+----------------+
|2025-01-01|820597.23         |62            |25          |24              |
|2025-01-02|1166289.6300000001|83            |25          |23              |
|2025-01-03|746560.4499999998 |70            |26          |23              |
|2025-01-04|847252.25         |84            |29          |27              |
|2025-01-05|878044.7299999999 |87            |28          |28              |
+----------+------------------+--------------+------------+----------------+
only showing top 5 rows

Category Sales:
+----------------+--------------------+--------------+------------+
|product_category|total_sales         |total_quantity|total_orders|
+----------------+--------------------+--------------+------------+
|GROCERY         |7.69846303399998E7  |6652      

In [12]:
# ===== GOLD DATA QUALITY CHECK =====

print("===== NULL CHECK: gold_sales =====")

gold_sales.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in gold_sales.columns
]).show()

print("===== DUPLICATE ORDER CHECK =====")

print(
    "Duplicate order_ids:",
    gold_sales.groupBy("order_id")
             .count()
             .filter(F.col("count") > 1)
             .count()
)

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 14, Finished, Available, Finished, False)

===== NULL CHECK: gold_sales =====
+--------+--------+-----------+--------+------------+------------+--------------+----------+----------+---------------+----------+----------+-------------+-------------+-------+------------+----------------+----------+----------+----------+
|quantity|order_id|customer_id|store_id|gross_amount|discount_pct|payment_method|unit_price|order_date|discount_amount|product_id|net_amount|customer_name|customer_city|segment|product_name|product_category|store_name|store_city|store_type|
+--------+--------+-----------+--------+------------+------------+--------------+----------+----------+---------------+----------+----------+-------------+-------------+-------+------------+----------------+----------+----------+----------+
|       0|       0|          0|       0|           0|           0|             0|         0|         0|              0|         0|         0|            0|            0|      0|           0|               0|         0|         0|         0|
+

In [13]:
# ===== BUSINESS KPI SANITY CHECK =====

print("===== GOLD BUSINESS KPI =====")

gold_sales.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.sum("gross_amount").alias("gross_sales"),
    F.sum("discount_amount").alias("total_discount"),
    F.sum("net_amount").alias("net_sales"),
    F.avg("net_amount").alias("avg_order_value")
).show()

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 15, Finished, Available, Finished, False)

===== GOLD BUSINESS KPI =====
+----------+-------------+-------------------+--------------------+--------------------+------------------+
|total_rows|unique_orders|        gross_sales|      total_discount|           net_sales|   avg_order_value|
+----------+-------------+-------------------+--------------------+--------------------+------------------+
|      9997|         9997|3.696616877399952E8|2.4441117359999985E7|3.4522057037999904E8|34532.416763028814|
+----------+-------------+-------------------+--------------------+--------------------+------------------+



In [14]:
# ===== OVERALL BUSINESS KPI =====

overall_kpi = gold_sales.agg(
    F.count("*").alias("total_transactions"),
    F.countDistinct("order_id").alias("total_orders"),
    F.countDistinct("customer_id").alias("total_customers"),
    F.sum("quantity").alias("total_quantity"),
    F.sum("gross_amount").alias("gross_sales"),
    F.sum("discount_amount").alias("total_discount"),
    F.sum("net_amount").alias("net_sales"),
    F.avg("net_amount").alias("average_order_value")
)

overall_kpi.show(truncate=False)

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 16, Finished, Available, Finished, False)

+------------------+------------+---------------+--------------+-------------------+--------------------+--------------------+-------------------+
|total_transactions|total_orders|total_customers|total_quantity|gross_sales        |total_discount      |net_sales           |average_order_value|
+------------------+------------+---------------+--------------+-------------------+--------------------+--------------------+-------------------+
|9997              |9997        |500            |30398         |3.696616877399958E8|2.4441117360000107E7|3.4522057037999946E8|34532.41676302886  |
+------------------+------------+---------------+--------------+-------------------+--------------------+--------------------+-------------------+



In [15]:
# ===== SAVE OVERALL KPI =====

overall_kpi.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("overall_kpi")

print("Overall KPI table saved successfully!")

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 17, Finished, Available, Finished, False)

Overall KPI table saved successfully!


In [16]:
# ===== VERIFY GOLD TABLES =====

spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 18, Finished, Available, Finished, False)

+------------------------------+--------------+-----------+
|namespace                     |tableName     |isTemporary|
+------------------------------+--------------+-----------+
|Retail360.Retail360_Silver.dbo|category_sales|false      |
|Retail360.Retail360_Silver.dbo|customers     |false      |
|Retail360.Retail360_Silver.dbo|daily_sales   |false      |
|Retail360.Retail360_Silver.dbo|gold_sales    |false      |
|Retail360.Retail360_Silver.dbo|overall_kpi   |false      |
|Retail360.Retail360_Silver.dbo|products      |false      |
|Retail360.Retail360_Silver.dbo|sales         |false      |
|Retail360.Retail360_Silver.dbo|segment_sales |false      |
|Retail360.Retail360_Silver.dbo|store_sales   |false      |
|Retail360.Retail360_Silver.dbo|stores        |false      |
+------------------------------+--------------+-----------+



In [17]:
# ===== SAVE GOLD TABLES =====

gold_sales.write.mode("overwrite").format("delta").saveAsTable("gold_sales")

daily_sales.write.mode("overwrite").format("delta").saveAsTable("daily_sales")

category_sales.write.mode("overwrite").format("delta").saveAsTable("category_sales")

store_sales.write.mode("overwrite").format("delta").saveAsTable("store_sales")

segment_sales.write.mode("overwrite").format("delta").saveAsTable("segment_sales")

overall_kpi.write.mode("overwrite").format("delta").saveAsTable("overall_kpi")

print("All Gold tables saved successfully!")

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 19, Finished, Available, Finished, False)

All Gold tables saved successfully!


In [18]:
# ===== VERIFY GOLD TABLES =====

spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 5f777679-00a2-4244-a5b6-e720a75e9894, 20, Finished, Available, Finished, False)

+------------------------------+--------------+-----------+
|namespace                     |tableName     |isTemporary|
+------------------------------+--------------+-----------+
|Retail360.Retail360_Silver.dbo|category_sales|false      |
|Retail360.Retail360_Silver.dbo|customers     |false      |
|Retail360.Retail360_Silver.dbo|daily_sales   |false      |
|Retail360.Retail360_Silver.dbo|gold_sales    |false      |
|Retail360.Retail360_Silver.dbo|overall_kpi   |false      |
|Retail360.Retail360_Silver.dbo|products      |false      |
|Retail360.Retail360_Silver.dbo|sales         |false      |
|Retail360.Retail360_Silver.dbo|segment_sales |false      |
|Retail360.Retail360_Silver.dbo|store_sales   |false      |
|Retail360.Retail360_Silver.dbo|stores        |false      |
+------------------------------+--------------+-----------+

